<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Document use boundaries

**[C05-M08-L02 · Document use boundaries](https://learning.nextia-ai.com/courses/ml/m08/document-use-boundaries/)** · C05, Machine Learning: From Problem to Reliable Model · Module 8, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** write a model card: what the model is for, what it must not be used for, its data, its results next to the baseline, its weak slices and its limits, and check that the card matches the artifact.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | The artifact in `model/`, from [Package artifacts](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/). The test report, from [Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/#test-once-with-evaluatepy). |
| **You do not need** | The local project. The setup below downloads the data and the course's `ticket_model.py`, `train.py` and `evaluate.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m08/document-use-boundaries/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M08-L02-document-use-boundaries/document-use-boundaries-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py`, `train.py`, `evaluate.py` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. Section 2 runs `train.py` and `evaluate.py` to make the evidence for the card.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py", "train.py", "evaluate.py"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A card for the rule "flag priority 1"

A box of medicine has a leaflet: what it is for, who must not take it, what was tested, and who made it. A model needs the same, because people use it far from the people who made it. One team lead already wants to rank support agents by the scores of their tickets, a use that the model never learned.

A **model card** is a short document that comes with a model. It says what the model is for, who uses it, what it must not be used for, what data trained it, how good it is, where it is weak, and who owns it.

Start with the baseline rule. Its card needs three numbers from the validation set (May): the share of tickets it flags, its precision and its recall.

> **Your turn.** Complete the cell. `rule_flags` is 1 for a priority 1 ticket. Set `rule_precision` (escalated among the flagged) and `rule_recall` (flagged among the escalated), each rounded to 3 decimals.

In [ ]:
from ticket_model import FEATURES, TARGET, load

valid = load("valid")
rule_flags = (valid["priority"] == 1).astype(int)
caught = int((rule_flags & valid[TARGET]).sum())
print("flagged:", rule_flags.sum(), "| escalated:", valid[TARGET].sum(), "| caught:", caught)

rule_precision = round(caught / rule_flags.sum(), 3)  # caught / flagged, rounded to 3 decimals
rule_recall = round(caught / valid[TARGET].sum(), 3)  # caught / escalated, rounded to 3 decimals
print(rule_precision, rule_recall)

In [ ]:
expect_hash('rule_precision and rule_recall', [rule_precision, rule_recall], '5a05bf63778ccd6e')

> **Check.** The first line says `flagged: 163 | escalated: 183 | caught: 61`. Check your two numbers by hand from these counts.

The rule's card:

| | |
|---|---|
| **What it does** | Flags every ticket with priority 1. |
| **For** | Sending tickets to the senior team when no model is available. |
| **Not for** | Deciding which customers matter more. |
| **Evidence (May)** | Flags 163 of 1,179 tickets (13.8%), with the precision and the recall that you computed. |
| **Limit** | Misses 2 of 3 escalations: every escalation with priority 2 or 3. |
| **Owner** | Grace |

## 2. Collect the evidence from the files

A card describes one exact artifact, so its numbers come from the files, not from memory. The next cell runs `train.py`, then `evaluate.py` once (it uses the test set; if the report exists already, the cell does not run it again).

In [ ]:
import sys
!"{sys.executable}" train.py
if not Path("reports/test_report.json").exists():
    !"{sys.executable}" evaluate.py
print("test report:", Path("reports/test_report.json").exists())

> **Check.** You should see `Threshold 0.19`, the test report as JSON (only the first time you run the cell), and `test report: True`.

Now the numbers for the card, as in the lesson's `card_numbers.py`:

In [ ]:
import json

info = json.load(open("model/model_info.json"))
test = json.load(open("reports/test_report.json"))
print("threshold:", info["threshold"], "| artifact:", info["artifact_sha256"][:16])
print("valid:", info["valid"])
print("test:", {k: test[k] for k in ("flag_rate", "precision", "recall", "roc_auc", "average_precision")})
print("rule on test:", test["baseline_priority_1"])

> **Check.** You should see `threshold: 0.19`, valid `'precision': 0.379, 'recall': 0.47`, test `'flag_rate': 0.196, 'precision': 0.357, 'recall': 0.446` and the rule on test `'flagged': 150, ..., 'recall': 0.35`. The artifact starts with `a4acd259a398e7c8` on macOS with the course's versions; on Colab or Kaggle it can differ.

## 3. Slices, with their counts

A card shows where the model is weaker. A rate from a small slice is unreliable, so each recall comes with its counts.

> **Your turn.** Complete the cell: set `recall_by_team` to a dict from team name to the recall at the threshold 0.19 on the validation set, rounded to 3 decimals. (Recall in a team: flagged escalations ÷ escalations.)

In [ ]:
import joblib

pipeline = joblib.load("model/escalation_model.joblib")
valid["flag"] = (pipeline.predict_proba(valid[FEATURES])[:, 1] >= 0.19).astype(int)
valid["caught"] = valid["flag"] & valid[TARGET]
counts = valid.groupby("team").agg(tickets=("flag", "size"), escalated=(TARGET, "sum"),
                                   flagged=("flag", "sum"), caught=("caught", "sum"))
print(counts)

recall_by_team = (counts["caught"] / counts["escalated"]).round(3).to_dict()
print(recall_by_team)

In [ ]:
expect_hash('recall_by_team', recall_by_team, '78c4370d3974d596')

> **Check.** The table has 5 teams. `account` has 166 tickets and 12 escalations, and the model catches only 1 of them. With 12 cases, that number is weak evidence; the card says so with the counts.

The same for the two smallest slices. Run the cell.

In [ ]:
for column, value in (("segment", "guest"), ("channel", "unknown")):
    part = valid[valid[column] == value]
    print(f"{column} = {value}: {len(part)} tickets, {part[TARGET].sum()} escalated, {part['flag'].sum()} flagged")

> **Check.** You should see `segment = guest: 9 tickets, 1 escalated, 1 flagged` and `channel = unknown: 22 tickets, 5 escalated, 5 flagged`. Too few to measure: the card says "too few to measure" instead of a recall of 1.000.

## 4. Write MODEL_CARD.md

The next cell writes Priya's card, the worked example of the [lesson page](https://learning.nextia-ai.com/courses/ml/m08/document-use-boundaries/#worked-example-priyas-model-card). Read it after you run the cell: each number is one that you saw above.

In [ ]:
%%writefile MODEL_CARD.md
# Model card: ticket escalation model

| | |
|---|---|
| Model | Logistic regression in a scikit-learn pipeline (`model/escalation_model.joblib`) |
| Artifact SHA-256 | `a4acd259a398e7c84517b98bf50d811197f90b0f2f1ea15f1f9e168ba3aff736` |
| Threshold | 0.19, chosen on the validation set for a review capacity of 20% |
| Card date | 2026-07-01 |
| Owners | Priya (data scientist): the model and this card. Grace (support lead): the decision and the review capacity. Mei (data engineer): the input data and the input checks. |
| Next review | 2026-10-01, or earlier when a retraining trigger fires |

## Intended use

- **Decision:** when a support ticket is created, decide if it goes straight to the senior team.
- **How:** `predict.py` scores each new ticket. A ticket with a score of at least 0.19 is flagged and routed to the senior team. Other tickets stay in the normal queue.
- **Users:** Larkfield's support team. Grace's senior team reviews the flagged tickets.
- **Prediction time:** when the ticket is created. The model uses only facts known at that moment.
- **Target:** `escalated_72h`, 1 if the ticket is escalated to a senior agent within 72 hours of creation.

## Not for

- **Staff performance reviews.** A score describes a ticket, not the agent who handled it.
- **Refusing, delaying or reducing service** to any customer. A ticket without a flag gets the normal service, never less.
- **Showing a score as "the chance" of escalation** to customers or staff. Since the policy change of 2026-05-01, the scores are too low (see *Limits*). This needs recalibration on recent data first.
- **Other shops, other languages, or other prediction times** (for example, after the first reply). The model learned Larkfield's tickets at creation only.
- **Ranking customers.** One prediction is for one ticket, not for a customer.

## Data

Synthetic data made for the course: it has no real people. One row is one ticket, with only the facts known at creation.

| Part | Tickets | Escalated | Dates |
|---|---|---|---|
| Train | 16,939 | 1,983 (11.7%) | 2024-07-01 to 2026-04-30 |
| Validation | 1,179 | 183 (15.5%) | May 2026 |
| Test | 1,002 | 157 (15.7%) | 2026-06-01 to 2026-06-27 |

Features (11): `channel`, `team`, `segment`, `region`, `priority`, `order_value`, `word_count`, `customer_tenure_days`, `prior_tickets_90d`, `created_hour`, `prior_escalations_90d`. The checksums of the data files are in `model/model_info.json`.

## Why this model

Priya compared logistic regression with decision trees, random forests and gradient boosting, with 5-fold cross-validation on the train set. The best boosting setting had an average precision of 0.3424, and logistic regression 0.3393. The difference, 0.003, is smaller than one standard error (0.008), so it is noise. By the one-standard-error rule, she chose the simpler model. Its coefficients show how each feature moves a score, so the owners can explain a flag and find problems in the inputs.

## Evaluation

The threshold was chosen on the validation set. The test set was used once, after every choice, with `python evaluate.py`.

| At threshold 0.19 | Validation (May) | Test (June) | Rule "flag priority 1" (test) |
|---|---|---|---|
| Flagged | 227 (19.3%) | 196 (19.6%) | 150 (15.0%) |
| Precision | 0.379 | 0.357 | 0.367 |
| Recall | 0.470 | 0.446 | 0.350 |
| ROC AUC | 0.752 | 0.749 | |
| Average precision | 0.391 | 0.345 | |

Target: flag at most 20% of new tickets and catch at least 40% of escalations. The model meets it on validation and test. The rule catches 35% on test (33.3% on validation).

Slices (validation set, threshold 0.19):

| Slice | Tickets | Escalated | Flagged | Recall |
|---|---|---|---|---|
| Team payment | 206 | 48 | 91 | 0.771 |
| Team warranty | 168 | 48 | 50 | 0.521 |
| Team delivery | 348 | 42 | 39 | 0.310 |
| Team returns | 291 | 33 | 40 | 0.303 |
| Team account | 166 | 12 | 7 | 0.083 |
| Priority 3 | 617 | 59 | 19 | 0.102 |
| Segment guest | 9 | 1 | 1 | too few to measure |
| Channel unknown | 22 | 5 | 5 | too few to measure |

## Limits

- **Scores are too low since 2026-05-01.** After the new warranty policy, the mean score on the validation set is 0.120, but 15.5% of tickets escalated. For scores from 0.2 to 0.3, the mean score is 0.241 and the observed rate is 0.315. The ranking still works, so the threshold for the capacity still works.
- **Precision is about 0.36 to 0.38.** About 2 of 3 flagged tickets do not escalate. The senior team must expect this.
- **Weak slices.** The model rarely flags `account` tickets and priority 3 tickets, so it misses most of their escalations.
- **Categories the model never saw.** A new channel (for example `social`) gets 0 in every channel column. The model still scores the ticket, but with less information. `predict.py` warns and counts these tickets.
- **Guests and new customers.** Guests have no customer history (202 guest tickets in train). Small slices like guests and `channel = unknown` have too few validation tickets to measure.
- **A hidden customer tendency.** Some repeat customers escalate more often, and no column shows it. This limits any model on these features.

## Ethical and practical notes

- A flag adds a review by a senior agent. No flag must never mean worse service.
- Coefficients and feature importance describe the model, not the causes of escalation.
- The owners check the monitoring signals every week (flag rate, mean score, unknown categories) and recall when the labels arrive.
- With real data, the features would include personal data about customers. Then privacy rules apply to the data, the artifact and the predictions.

## How to reproduce

```sh
python -m pip install -r requirements.txt
python get_data.py
python train.py
python predict.py --check
```

`python train.py` prints `Trained on 16939 tickets. Threshold 0.19 for a review capacity of 20%.` `python predict.py --check` prints `Check passed: 20 scores are identical after reloading`.

The next cell checks that the card has every section of the lesson. Run it. You should see `All 8 sections are there.`

In [ ]:
card = Path("MODEL_CARD.md").read_text(encoding="utf-8")
sections = ["Intended use", "Not for", "Data", "Why this model", "Evaluation", "Limits", "Ethical and practical notes", "How to reproduce"]
missing = [s for s in sections if f"## {s}" not in card]
print("Missing:", missing) if missing else print(f"All {len(sections)} sections are there.")

## 5. Your turn: Omar's request

Omar wants each confirmation email to say: "There is a 24% chance that a senior agent will handle your ticket." The 24% is the ticket's score. The next cell computes the evidence: for validation tickets with a score from 0.2 to 0.3, the mean score and the share that escalated.

> **Predict.** Is the share that escalated higher, lower or about the same as the mean score?

In [ ]:
scores = pipeline.predict_proba(valid[FEATURES])[:, 1]
band = (scores > 0.2) & (scores <= 0.3)
print("tickets:", band.sum(), "| mean score:", round(scores[band].mean(), 3),
      "| escalated:", round(valid.loc[band, TARGET].mean(), 3))

> **Check.** You should see `tickets: 111 | mean score: 0.241 | escalated: 0.315`.

> **Your turn.** Write your answer to Omar in the next cell (double-click to edit): yes or no, and why, with the numbers. Then compare with the answer on the lesson page.

*Your answer here:*

## 6. Failure case: a card that does not match the artifact

The card was written once, by hand. If the artifact changes after it, the card is wrong, and nobody sees an error. The next cell writes `check_card.py`: it checks that the card names the model file's SHA-256 and the threshold of `model_info.json`.

In [ ]:
%%writefile check_card.py
import json
from pathlib import Path

card = Path("MODEL_CARD.md").read_text(encoding="utf-8")
info = json.loads(Path("model/model_info.json").read_text(encoding="utf-8"))
problems = []
if info["artifact_sha256"] not in card:
    problems.append("the card names another model file")
if f"| Threshold | {info['threshold']}," not in card:
    problems.append(f"the card does not give the threshold {info['threshold']}")
print("The card matches the artifact." if not problems else "The card does not match: " + "; ".join(problems) + ".")

Run it for the course's artifact. You should see `The card matches the artifact.`

In [ ]:
!"{sys.executable}" check_card.py

Now Tomás's mistake: the capacity of 15% from Lesson 1, and he forgets to set it back.

> **Predict.** The pipeline is the same; only the threshold changes. Will the model file's SHA-256 change? What will `check_card.py` say?

In [ ]:
text = Path("train.py").read_text(encoding="utf-8")
Path("train.py").write_text(text.replace("REVIEW_CAPACITY = 0.20", "REVIEW_CAPACITY = 0.15"), encoding="utf-8")
!"{sys.executable}" train.py
print("artifact:", json.load(open("model/model_info.json"))["artifact_sha256"][:16])
!"{sys.executable}" check_card.py

> **Check.** You should see `Threshold 0.22`, the same artifact `a4acd259a398e7c8` (on the course's versions), and `The card does not match: the card does not give the threshold 0.22.` The model file did not change, but the artifact's behaviour did: the real recall on May is now 0.41, not the card's 0.470.

Set the capacity back and make the course's artifact again. You should see `Threshold 0.19` and `The card matches the artifact.`

In [ ]:
Path("train.py").write_text(text, encoding="utf-8")
!"{sys.executable}" train.py
!"{sys.executable}" check_card.py

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m08/document-use-boundaries/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

A model card says what the model is for, what it is not for (with reasons), its data, why this model, its results next to the baseline, its slices with counts, its limits, and its owners and review date. Its numbers come from the files, and a short check proves that the card matches the artifact.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Plan production checks](https://learning.nextia-ai.com/courses/ml/m08/plan-production-checks/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m08/document-use-boundaries/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.